## Before running this notebook:
If you don't have Xolotl installed yet, refer to https://github.com/ORNL-Fusion/xolotl/wiki/Build-Configuration#quick-start
<br>
<br>
Copy the parameter file ```params_system_PSI_8.json``` and temperature file ```temp_system_PSI_8.dat``` (from ./xolotl/benchmarks/) in your build folder.
<br>
<br>
(Serial) Run the following command in your terminal within the build directory:


```bash
./xolotl params_system_PSI_8.json
```
(Parallel) Run the following command in your terminal:

```bash
mpiexec -n 4 ./xolotl params_system_PSI_8.json
```
For more information on running Xolotl, see the wiki: https://github.com/ORNL-Fusion/xolotl/wiki/Running-Xolotl

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import h5py
from IPython.display import HTML


Modules to load. Most important for extracting data from Xolotl is h5py.

In [2]:
colorVec = ['k', 'orange', 'b', 'm', 'r', 'g']

def readTridyn(fileName, types=[['He',1], ['V',2], ['I',3]], temp_col=4):
    with h5py.File(fileName, 'r') as f:
        concDset = f['concs'][:]

    depth = concDset[:, 0]
    concs = []

    for species in types:
        col_idx = species[1]
        concs.append(concDset[:, col_idx])
    temp_profile = concDset[:, temp_col]
    if np.allclose(temp_profile, temp_profile[0]):
        temperature = temp_profile[0]
    else:
        temperature = np.mean(temp_profile)

    return depth, concs, temperature

Read depth and concentration columns from TRIDYN h5 files. This example's simulation only has He, but obtaining V or I concentration would utilize the same method.

In [3]:
folder = r'C:\Users\mrcon\OneDrive\Desktop\research\xolotl\TDS'
start_num = 218
end_num   = 868
step_num  = 10

types = [['He',1], ['V',2], ['I',3]]
output_gif = 'tridyn_animation.gif'
fps = 3               # frames per second

file_numbers = list(range(start_num, end_num + 1, step_num))
file_list = [os.path.join(folder, f'TRIDYN_{n}.h5') for n in file_numbers]
valid_files = []
valid_nums = []


Start and end steps may need to be changed depending on how you modify your input file. 

In [ ]:
for f, n in zip(file_list, file_numbers):
    if os.path.exists(f):
        valid_files.append(f)
        valid_nums.append(n)
    else:
        print(f"Warning: file not found, skipping: {f}")

if not valid_files:
    raise FileNotFoundError("No valid TRIDYN files found.")

In [5]:
all_data = []
xmin, xmax = np.inf, -np.inf
ymin, ymax = np.inf, -np.inf

for f in valid_files:
    depth, concs, temperature = readTridyn(f, types=types)
    all_data.append((depth, concs, temperature))

    xmin = min(xmin, np.min(depth))
    xmax = max(xmax, np.max(depth))
    for c in concs:
        ymin = min(ymin, np.min(c))
        ymax = max(ymax, np.max(c))

# Optional padding for nicer plot limits
xpad = 0.05 * (xmax - xmin) if xmax > xmin else 1.0
ypad = 0.05 * (ymax - ymin) if ymax > ymin else 1.0

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

lines = []
for i, species in enumerate(types):
    line, = ax.plot([], [], lw=4, color=colorVec[i], label=species[0])
    lines.append(line)

title = ax.set_title("", fontsize=25)
ax.set_xlabel("Depth [nm]", fontsize=25)
ax.set_ylabel("Concentration [atoms/nm3]", fontsize=25)
ax.tick_params(axis='both', which='major', labelsize=20)
ax.tick_params(axis='both', which='minor', labelsize=20)
ax.legend(loc='best', handlelength=4, ncol=3, fontsize=20)
ax.set_xlim([0.0, 1000.0])
ax.set_ylim([1.0e-12, 1.0e-4])

#ax.set_xscale('log')
ax.set_yscale('log')

In [7]:
def init():
    for line in lines:
        line.set_data([], [])
    title.set_text("")
    return lines + [title]

def update(frame_idx):
    depth, concs, temperature = all_data[frame_idx]

    for i, line in enumerate(lines):
        line.set_data(depth, concs[i])

    title.set_text(f"TRIDYN step {valid_nums[frame_idx]}, T = {temperature:.1f} K")
    return lines + [title]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(all_data),
    init_func=init,
    blit=True
)

# Save GIF
ani.save(output_gif, writer=animation.PillowWriter(fps=fps))

print(f"Saved GIF: {output_gif}")
HTML(ani.to_jshtml())

Saved GIF: tridyn_animation.gif
